# FactoryBench Q&A Generation
This notebook generates Q&A pairs focused on **Pearl's Causal Levels**.

- Reference: (E. Bareinboim, J. D. Correa, D. Ibeling, and T. Icard, “On Pearl’s hierarchy and the
foundations of causal inference,” Columbia University, New York, NY, Tech. Rep.
R-60, 2020, First Version: July 2020. Last Revision: Mar 2021. [Online]. Available:
https://causalai.net/r60.pdf. )

> 🪸 NOTES:
> - Implemented level 1 (observing) questions and answers
> - ABB IRB2600 robot

In [1]:

from dataclasses import dataclass
from typing import List, Optional, Dict, Any
import random
import os
import sys
import json
import glob
import json

sys.path.append(os.getcwd())

from question_templates.observation_templates import ObservationTemplate
from question_templates.telemetry_templates import TelemetryTemplate
sys.path.append(os.path.abspath(".."))
from factorybench.data.loader_tl import load_telemetry_literacy


OUTPUT_DIR = "generated_qa"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Definition of Robots

In [2]:
@dataclass
class RobotSpec:
    model: str
    manufacturer: str
    family: str
    dof: int
    payload_kg: float
    reach_m: float
    joint_names: List[str]
    urdf_path: Optional[str] = None
    manual_path: Optional[str] = None

# ABB IRB2600
abb_irb2600 = RobotSpec(
    model="IRB2600",
    manufacturer="ABB",
    family="IRB",
    dof=6,
    payload_kg=20.0,
    reach_m=1.65,
    joint_names=["joint_1", "joint_2", "joint_3", "joint_4", "joint_5", "joint_6"],
    manual_path="manuals/ABB_IRB2600/3HAC035504 PM IRB 2600-en.pdf"
)

# KUKA KR10 R1100
kuka_kr10 = RobotSpec(
    model="KR10 R1100",
    manufacturer="KUKA",
    family=None,
    dof=None,
    payload_kg=None,
    reach_m=None,
    joint_names=None,
    manual_path=None
    # TODO: Complete it
)

# UR3
ur3 = RobotSpec(
    model="UR3",
    manufacturer="Universal Robots",
    family=None,
    dof=None,
    payload_kg=None,
    reach_m=None,
    joint_names=None,
    manual_path=None
    # TODO: Complete it
)

# ABB YUMI
abb_yumi = RobotSpec(
    model="YUMI",
    manufacturer="ABB",
    family=None,
    dof=None,
    payload_kg=None,
    reach_m=None,
    joint_names=None,
    manual_path=None
    # TODO: Complete it
)

ROBOTS = [abb_irb2600, kuka_kr10, ur3, abb_yumi]

CURRENT_ROBOT = abb_irb2600

## Load Synthetic Datasets (Level 1: Basic Statistics & Point Queries)
- `basic_statistics.json`: temporal series with basic stats (mean, std, min, max) 
- `step_functions.json`: temporal series with step changes (abrupt changes in value)

In [3]:
try:
    basic_stats = load_telemetry_literacy(path="../datasets/basic_statistics.json")
    step_funcs = load_telemetry_literacy(path="../datasets/step_functions.json")
    print(f"Loaded datasets: {len(basic_stats)} basic stats, {len(step_funcs)} step functions")
except Exception as e:
    print(f"Error loading datasets: {e}")
    basic_stats = []
    step_funcs = []


Loaded datasets: 10 basic stats, 15 step functions


In [4]:
def generate_observation_qa(robot: RobotSpec, q_type: str, data_sample: Optional[Dict] = None, question_type: str = "open") -> Dict[str, Any]:
    """Generates questions about the current observed state using Pearl's Level 1 templates."""
    
    if data_sample:
        tt = TelemetryTemplate()
        
        # Statistical Questions (Level 1)
        if q_type == "stats":
            stats = data_sample.get("statistics", {})
            metric = random.choice(["mean", "std", "min", "max"])
            if metric in stats:
                result = tt.get_stats_question(metric, stats[metric], format=question_type)
                if result:
                    # Enrich with robot context
                    result["robot"] = robot.model
                    result["verification"] = {
                        "source": "synthetic_dataset",
                        "sample_id": data_sample.get("id")
                    }
                    return result
        
        # Point Queries (Level 1) - are generated by randomly sampling a timestamp and value from the data sample, then asking a question about that specific point
        elif q_type == "point":
            timestamps = data_sample.get("timestamps", [])
            values = data_sample.get("values", [])
            if timestamps and values:
                idx = random.randint(0, len(timestamps)-1)
                result = tt.get_point_question(timestamps[idx], values[idx], format=question_type)
                if result:
                    result["robot"] = robot.model
                    result["verification"] = {
                        "source": "synthetic_dataset",
                        "sample_id": data_sample.get("id")
                    }
                    return result

    return None


## Level 1 Q&A pairs (Observation)
Generating questions based on passive observation of the system state.

- **Associational (Seeing) P(y|x):** It is based on passive observation to identify statistical
correlations and answer questions about what an observation reveals about
a specific event. It includes supervised and unsupervised machine learning tasks.
The typical question are "What is?" and "How would seeing X change my belief in
Y?".


In [5]:
# Generate Level 1 Q&A pairs (Observation)
# 20 multiple choice, 20 true/false, 20 open-ended for each dataset type (stats, point)
datasets_map = {
    "stats": basic_stats,
    "point": step_funcs
}

question_types = ["open", "mc", "tf"]
target_count = 20

for q_type, dataset in datasets_map.items():
    if not dataset: continue
    
    for question_type in question_types:
        print(f"Generating {target_count} {question_type} questions for {q_type}...")
        count = 0
        sample_idx = 0
        
        while count < target_count:
            sample = dataset[sample_idx % len(dataset)]
            sample_idx += 1
            
            # Generate a question for this sample with specific format
            qa_pair = generate_observation_qa(CURRENT_ROBOT, q_type, data_sample=sample, question_type=question_type)
            
            if qa_pair:
                robot_name = CURRENT_ROBOT.model.replace(" ", "_")
                # Unique filename per count to avoid overwrite
                filename = f"{OUTPUT_DIR}/qa_{robot_name}_{q_type}_{question_type}_{count+1}.json"
                
                with open(filename, 'w', encoding='utf-8') as f:
                    json.dump(qa_pair, f, indent=2)
                
                    print(f"Saved {filename}")
                
                count += 1


Generating 20 open questions for stats...
Saved generated_qa/qa_IRB2600_stats_open_1.json
Saved generated_qa/qa_IRB2600_stats_open_2.json
Saved generated_qa/qa_IRB2600_stats_open_3.json
Saved generated_qa/qa_IRB2600_stats_open_4.json
Saved generated_qa/qa_IRB2600_stats_open_5.json
Saved generated_qa/qa_IRB2600_stats_open_6.json
Saved generated_qa/qa_IRB2600_stats_open_7.json
Saved generated_qa/qa_IRB2600_stats_open_8.json
Saved generated_qa/qa_IRB2600_stats_open_9.json
Saved generated_qa/qa_IRB2600_stats_open_10.json
Saved generated_qa/qa_IRB2600_stats_open_11.json
Saved generated_qa/qa_IRB2600_stats_open_12.json
Saved generated_qa/qa_IRB2600_stats_open_13.json
Saved generated_qa/qa_IRB2600_stats_open_14.json
Saved generated_qa/qa_IRB2600_stats_open_15.json
Saved generated_qa/qa_IRB2600_stats_open_16.json
Saved generated_qa/qa_IRB2600_stats_open_17.json
Saved generated_qa/qa_IRB2600_stats_open_18.json
Saved generated_qa/qa_IRB2600_stats_open_19.json
Saved generated_qa/qa_IRB2600_stats_

## Check the correctiveness of the Q&A pairs

In [6]:
def verify_qa_integrity():
    files = glob.glob(f"{OUTPUT_DIR}/*.json")
    print(f"Verifying {len(files)} generated files in {OUTPUT_DIR}...")
    errors = []
    passed = 0
    
    for f_path in files:
        try:
            with open(f_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # structure check
            if "question" not in data or "answer" not in data:
                errors.append(f"{f_path}: Missing keys")
                continue
            
            q = data["question"]
            a = data["answer"]
            
            # multiple choice consistency check
            if "options" in q:
                opt_str = [str(o) for o in q["options"]]
                if str(a["value"]) not in opt_str:
                    try:
                        # convert to 4 decimal float for comparison
                        val_float = round(float(a["value"]), 4)
                        opts_float = [round(float(o), 4) for o in q["options"]]
                        if val_float not in opts_float:
                            errors.append(f"{f_path}: MC Answer {a['value']} not found in options {q['options']}")
                    except:
                        errors.append(f"{f_path}: MC Answer {a['value']} not found in options {q['options']}")
            
            # true/false consistency check
            if "level1_stats_" in q["type"] and "_tf" in q["type"]:
                if a["value"] not in ["True", "False"]:
                    errors.append(f"{f_path}: TF Answer {a['value']} is not True/False")

            # open-ended answer check (basic non-empty)
            if "open" in q["type"]:
                if not a["value"] or (isinstance(a["value"], str) and a["value"].strip() == ""):
                    errors.append(f"{f_path}: Open-ended answer is empty")
            
            passed += 1
        except Exception as e:
            errors.append(f"{f_path}: Read error {e}")

    if not errors:
        print(f"✅ All {passed} files passed integrity checks.")
    else:
        print(f"❌ Found {len(errors)} errors:")
        for e in errors[:10]: print(e)

verify_qa_integrity()


Verifying 120 generated files in generated_qa...
✅ All 120 files passed integrity checks.
